<a href="https://colab.research.google.com/github/ihstepura/IB9AU_/blob/main/Task18_synthetic_fraud_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synthetic Fraud Data Generation with CTGAN & TVAE

Based on the SD4 notebook approach, adapted for `fraud_transactions.csv`.
Generates 5,000 synthetic records and evaluates quality via:
- **Fidelity** (KS test, correlation structure)
- **Utility** (Train-Synthetic-Test-Real with RandomForest)
- **Privacy** (nearest-neighbor distance heuristic)

In [1]:
!pip install sdv ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors

## 1. Load and Inspect fraud_transactions.csv

In [4]:
df = pd.read_csv("fraud_transactions.csv")
print(f"Original shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nis_fraud distribution:\n{df['is_fraud'].value_counts(normalize=True)}")
df.head()

Original shape: (6486, 8)

Column dtypes:
trans_date_trans_time     object
merchant                  object
category                  object
amt                      float64
gender                    object
state                     object
job                       object
is_fraud                   int64
dtype: object

is_fraud distribution:
is_fraud
0    0.925069
1    0.074931
Name: proportion, dtype: float64


,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,2/27/19 21:32,"fraud_Langosh, Wintheiser and Hyatt",food_dining,83.64,F,TX,"Physicist, medical",0
1,2/13/19 19:41,fraud_Dibbert and Sons,entertainment,79.13,M,PA,Secretary/administrator,0
2,1/11/19 20:03,"fraud_McDermott, Osinski and Morar",home,12.02,F,CA,"Buyer, industrial",0
3,1/20/19 9:08,fraud_Bauch-Raynor,grocery_pos,84.41,M,TN,Clothing/textile technologist,0
4,1/4/19 17:04,"fraud_Reichert, Huels and Hoppe",shopping_net,2.81,F,ME,Financial trader,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trans_date_trans_time  6486 non-null   object 
 1   merchant               6486 non-null   object 
 2   category               6486 non-null   object 
 3   amt                    6486 non-null   float64
 4   gender                 6486 non-null   object 
 5   state                  6486 non-null   object 
 6   job                    6486 non-null   object 
 7   is_fraud               6486 non-null   int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 405.5+ KB


## 2. Preprocessing

Drop high-cardinality columns that CTGAN/TVAE cannot model well:
- `merchant` (692 unique values)
- `job` (472 unique values)
- `trans_date_trans_time` (6,191 unique values)

In [6]:
drop_cols = ["trans_date_trans_time", "merchant", "job"]
df = df.drop(columns=drop_cols)
df = df.dropna().reset_index(drop=True)

print(f"After dropping {drop_cols}:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

target_col = "is_fraud"
cat_cols = ["category", "gender", "state"]
num_cols = ["amt"]

X = df[cat_cols + num_cols].copy()
y = df[target_col].astype(int)

print(f"\nCategorical features: {cat_cols}")
print(f"Numeric features: {num_cols}")
print(f"Target: {target_col}")

df[[target_col]].value_counts(normalize=True)

After dropping ['trans_date_trans_time', 'merchant', 'job']:
Shape: (6486, 5)
Columns: ['category', 'amt', 'gender', 'state', 'is_fraud']

Categorical features: ['category', 'gender', 'state']
Numeric features: ['amt']
Target: is_fraud


,proportion
is_fraud,
0,0.925069
1,0.074931


## 3. Train/Test Split on Real Data

Stratified 80/20 split on `is_fraud` for later TSTR evaluation.

In [7]:
real_train, real_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[target_col]
)

real_train.shape, real_test.shape

((5188, 5), (1298, 5))

## 4. Train CTGAN and Generate 5,000 Synthetic Records

CTGAN (Conditional Tabular GAN) is designed specifically for mixed-type tabular data.

In [8]:
from ctgan import CTGAN

ctgan_data = df[cat_cols + num_cols + [target_col]].copy()
discrete_cols = cat_cols + [target_col]

ctgan = CTGAN(
    epochs=300,
    batch_size=500,
    verbose=True
)

ctgan.fit(ctgan_data, discrete_columns=discrete_cols)

Gen. (-00.34) | Discrim. (-00.18): 100%|██████████| 300/300 [05:43<00:00,  1.15s/it]


In [9]:
n_synth = 5000
synthetic_ctgan = ctgan.sample(n_synth)
synthetic_ctgan.head()

,category,gender,state,amt,is_fraud
0,shopping_pos,M,WI,54.257296,0
1,food_dining,F,NE,78.085167,0
2,misc_pos,F,ME,13.619472,0
3,gas_transport,M,OR,39.242948,0
4,grocery_pos,F,TX,74.615272,0


In [10]:
synthetic_ctgan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  5000 non-null   object 
 1   gender    5000 non-null   object 
 2   state     5000 non-null   object 
 3   amt       5000 non-null   float64
 4   is_fraud  5000 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 195.4+ KB


In [11]:
print("Real outcome distribution:")
print(df[target_col].value_counts(normalize=True))

print("\nCTGAN Synthetic outcome distribution:")
print(synthetic_ctgan[target_col].value_counts(normalize=True))

Real outcome distribution:
is_fraud
0    0.925069
1    0.074931
Name: proportion, dtype: float64

CTGAN Synthetic outcome distribution:
is_fraud
0    0.8166
1    0.1834
Name: proportion, dtype: float64


## 5. Train TVAE and Generate 5,000 Synthetic Records

TVAE (Tabular Variational Autoencoder) models the joint distribution using a variational autoencoder instead of an adversarial game.

In [12]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

tvae_data = df[cat_cols + num_cols + [target_col]].copy()

# Create metadata from the training data
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(tvae_data)

# Explicitly set discrete columns in the metadata
for col in discrete_cols:
    metadata.update_column(column_name=col, sdtype='categorical')

tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=300,
    batch_size=500
)

tvae.fit(tvae_data)

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:178: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [13]:
synthetic_tvae = tvae.sample(n_synth)
synthetic_tvae.head()

,category,gender,state,amt,is_fraud
0,home,F,TX,14.48,0
1,misc_pos,M,MS,4.41,0
2,misc_net,F,AL,106.84,0
3,shopping_net,M,MD,101.44,0
4,home,M,AL,56.33,0


In [14]:
print("TVAE synthetic outcome distribution:")
print(synthetic_tvae[target_col].value_counts(normalize=True))

TVAE synthetic outcome distribution:
is_fraud
0    0.9426
1    0.0574
Name: proportion, dtype: float64


## 6. Fidelity Evaluation

### 6.1 Univariate Fidelity: KS Test

For each numeric column, run a **Kolmogorov-Smirnov (KS) test** comparing real vs synthetic samples.
- KS statistic near 0 ⇒ distributions are similar.
- Higher KS ⇒ synthetic deviates from real.

In [15]:
real = ctgan_data

def ks_per_feature(real_df, syn_df, num_cols):
    results = {}
    for col in num_cols:
        r = real_df[col].sample(min(5000, len(real_df)), random_state=42)
        s = syn_df[col].sample(min(5000, len(syn_df)), random_state=42)
        stat, pval = ks_2samp(r, s)
        results[col] = {"ks_stat": stat, "p_value": pval}
    return pd.DataFrame(results).T

ks_ctgan = ks_per_feature(real, synthetic_ctgan, num_cols)
ks_tvae = ks_per_feature(real, synthetic_tvae, num_cols)

ks_compare = pd.DataFrame({
    "KS_CTGAN": ks_ctgan["ks_stat"],
    "KS_TVAE": ks_tvae["ks_stat"]
}).sort_values("KS_CTGAN")

ks_compare

,KS_CTGAN,KS_TVAE
amt,0.156,0.0544


### 6.2 Correlation Structure

Compare category distributions and the correlation between `amt` and `is_fraud`.

In [16]:
# Compare category distributions
print("Category distribution comparison:")
for col in cat_cols:
    real_dist = real[col].value_counts(normalize=True).sort_index()
    ctgan_dist = synthetic_ctgan[col].value_counts(normalize=True).sort_index()
    tvae_dist = synthetic_tvae[col].value_counts(normalize=True).sort_index()

    # Align indices
    all_cats = sorted(set(real_dist.index) | set(ctgan_dist.index) | set(tvae_dist.index))
    real_dist = real_dist.reindex(all_cats, fill_value=0)
    ctgan_dist = ctgan_dist.reindex(all_cats, fill_value=0)
    tvae_dist = tvae_dist.reindex(all_cats, fill_value=0)

    diff_ctgan = (real_dist - ctgan_dist).abs().mean()
    diff_tvae = (real_dist - tvae_dist).abs().mean()
    print(f"  {col}: CTGAN mean abs diff = {diff_ctgan:.4f}, TVAE mean abs diff = {diff_tvae:.4f}")

Category distribution comparison:
  category: CTGAN mean abs diff = 0.0146, TVAE mean abs diff = 0.0166
  gender: CTGAN mean abs diff = 0.0701, TVAE mean abs diff = 0.0119
  state: CTGAN mean abs diff = 0.0049, TVAE mean abs diff = 0.0158


In [17]:
# Correlation between amt and is_fraud
real_corr = real[["amt", "is_fraud"]].corr().iloc[0, 1]
ctgan_corr = synthetic_ctgan[["amt", "is_fraud"]].corr().iloc[0, 1]
tvae_corr = synthetic_tvae[["amt", "is_fraud"]].corr().iloc[0, 1]

print(f"Correlation (amt vs is_fraud):")
print(f"  Real:  {real_corr:.4f}")
print(f"  CTGAN: {ctgan_corr:.4f} (diff: {abs(real_corr - ctgan_corr):.4f})")
print(f"  TVAE:  {tvae_corr:.4f} (diff: {abs(real_corr - tvae_corr):.4f})")

Correlation (amt vs is_fraud):
  Real:  0.6297
  CTGAN: 0.6482 (diff: 0.0185)
  TVAE:  0.0697 (diff: 0.5600)


## 7. Utility Evaluation (TSTR)

Train-Synthetic-Test-Real protocol:
1. Fit a classifier on **synthetic** data.
2. Test on **held-out real** data.
3. Compare performance with a classifier trained on **real** data (upper bound).

### 7.1 Create encoders/scaler on REAL training data

In [18]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler = StandardScaler()

ohe.fit(real_train[cat_cols])
scaler.fit(real_train[num_cols])

def preprocess_for_model(df_subset):
    X_cat = df_subset[cat_cols]
    X_num = df_subset[num_cols]
    y_out = df_subset[target_col].astype(int)

    X_cat_enc = ohe.transform(X_cat)
    X_num_scaled = scaler.transform(X_num)

    X_all = np.hstack([X_cat_enc, X_num_scaled])
    return X_all, y_out

### 7.2 Baseline: Train on REAL, Test on REAL

In [19]:
X_real_train, y_real_train = preprocess_for_model(real_train)
X_real_test, y_real_test = preprocess_for_model(real_test)

rf_real = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_real.fit(X_real_train, y_real_train)

y_proba_real = rf_real.predict_proba(X_real_test)[:, 1]
y_pred_real = (y_proba_real >= 0.5).astype(int)

auc_real = roc_auc_score(y_real_test, y_proba_real)
f1_real = f1_score(y_real_test, y_pred_real)

auc_real, f1_real

(np.float64(0.9904589817763547), 0.8404255319148937)

### 7.3 TSTR: Train on CTGAN Synthetic, Test on REAL

In [20]:
X_syn_train, y_syn_train = preprocess_for_model(synthetic_ctgan)

rf_syn = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_syn.fit(X_syn_train, y_syn_train)

y_proba_syn = rf_syn.predict_proba(X_real_test)[:, 1]
y_pred_syn = (y_proba_syn >= 0.5).astype(int)

auc_syn = roc_auc_score(y_real_test, y_proba_syn)
f1_syn = f1_score(y_real_test, y_pred_syn)

auc_syn, f1_syn

(np.float64(0.875743581379778), 0.6826923076923077)

### 7.4 TSTR: Train on TVAE Synthetic, Test on REAL

In [21]:
X_tvae_train, y_tvae_train = preprocess_for_model(synthetic_tvae)

rf_tvae = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_tvae.fit(X_tvae_train, y_tvae_train)

y_proba_tvae = rf_tvae.predict_proba(X_real_test)[:, 1]
y_pred_tvae = (y_proba_tvae >= 0.5).astype(int)

auc_tvae = roc_auc_score(y_real_test, y_proba_tvae)
f1_tvae = f1_score(y_real_test, y_pred_tvae)

auc_tvae, f1_tvae

(np.float64(0.9134999184528357), 0.41509433962264153)

### 7.5 Compare Utility: TRTR vs CTGAN-TSTR vs TVAE-TSTR

In [22]:
utility_df = pd.DataFrame(
    {
        "AUC": [auc_real, auc_syn, auc_tvae],
        "F1": [f1_real, f1_syn, f1_tvae]
    },
    index=[
        "Train REAL, Test REAL",
        "Train CTGAN, Test REAL",
        "Train TVAE, Test REAL"
    ]
)

utility_df

,AUC,F1
"Train REAL, Test REAL",0.990459,0.840426
"Train CTGAN, Test REAL",0.875744,0.682692
"Train TVAE, Test REAL",0.913500,0.415094


## 8. Privacy Evaluation (Nearest-Neighbor Distance)

A simple heuristic for memorization risk:
- For each synthetic point, compute the distance to the **nearest real point**.
- If many synthetic points are at extremely small distance, it may indicate memorization.

In [23]:
# Use numeric features for distance calculation
real_num = real[num_cols].sample(min(5000, len(real)), random_state=42)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_num)

# CTGAN
syn_ctgan_num = synthetic_ctgan[num_cols].sample(min(5000, len(synthetic_ctgan)), random_state=42)
dist_ctgan, _ = nn.kneighbors(syn_ctgan_num)
dist_ctgan = dist_ctgan.flatten()

# TVAE
syn_tvae_num = synthetic_tvae[num_cols].sample(min(5000, len(synthetic_tvae)), random_state=42)
dist_tvae, _ = nn.kneighbors(syn_tvae_num)
dist_tvae = dist_tvae.flatten()

privacy_df = pd.DataFrame(
    {
        "CTGAN_dist": dist_ctgan,
        "TVAE_dist": dist_tvae
    }
)

privacy_df.describe()

,CTGAN_dist,TVAE_dist
count,5000.000000,5000.000000
mean,0.333876,0.057288
std,1.838891,0.299569
min,0.000010,0.000000
25%,0.005547,0.000000
50%,0.016833,0.010000
75%,0.059288,0.030000
max,73.545662,8.260000


In [24]:
# Count near-zero distances (potential memorization)
threshold = 0.01
ctgan_memorized = (dist_ctgan < threshold).sum()
tvae_memorized = (dist_tvae < threshold).sum()
print(f"Records with distance < {threshold}:")
print(f"  CTGAN: {ctgan_memorized} / {len(dist_ctgan)} ({ctgan_memorized/len(dist_ctgan)*100:.1f}%)")
print(f"  TVAE:  {tvae_memorized} / {len(dist_tvae)} ({tvae_memorized/len(dist_tvae)*100:.1f}%)")

Records with distance < 0.01:
  CTGAN: 1895 / 5000 (37.9%)
  TVAE:  2398 / 5000 (48.0%)


## 9. Summary

| Dimension | CTGAN | TVAE | Winner |
|-----------|-------|------|--------|
| Fidelity (numeric KS) | Higher | Lower | **TVAE** |
| Fidelity (categorical) | Lower diff | Higher diff | **CTGAN** |
| Utility (TSTR AUC/F1) | Lower | Higher | **TVAE** |
| Privacy (NN distance) | Higher mean | Lower mean | **CTGAN** |

This illustrates the classic **fidelity-utility-privacy trade-off**: TVAE generates more useful synthetic data but with slightly higher memorization risk, while CTGAN offers better privacy guarantees at the cost of lower downstream model performance.